# 6. Greyhound ELO Ratings

This notebook implements a greyhound-specific ELO rating system at the **race-row** level.

Design choices in this first version:

- Multi-dog races are decomposed into pairwise matchups.
- The actual score uses a margin-aware time-gap bonus when `resultRunTime` is available.
- Each dog keeps an **overall** ELO and a **distance-band** ELO.
- Ratings decay back toward the population mean between races.
- New dogs can optionally be seeded from starting-price implied probability.

This gives us a robust sequential rating that can later be extended with track-specific or trap-specific adjustments.

# NB: 

Important to note that the implementation of ELO rating into the model inputs will make the prediction phase on real races harder.
We will need to update the whole dataset to get all infos of all the dogs in the past dog races. Which means that we need all the new available data each time.

In [1]:
import math
from collections import defaultdict

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

## 6.1 Load and prepare the race rows

We use the cleaned intermediate dataset and keep only the columns needed for sequential race-level rating updates.

In [2]:
elo_cols = [
    "dogId",
    "raceId",
    "raceDate",
    "raceTime",
    "trackName",
    "trapNumber",
    "raceDistance",
    "raceClass",
    "resultPosition",
    "resultRunTime",
    "numeratorSP",
    "denominatorSP",
]

dog_infos = pd.read_csv("../data/intermediate/04_dog_infos_cleaned.csv", usecols=elo_cols)

dog_infos["raceDate"] = pd.to_datetime(dog_infos["raceDate"], errors="coerce")
dog_infos["raceTime"] = dog_infos["raceTime"].fillna("00:00:00")
dog_infos["raceDateTime"] = pd.to_datetime(
    dog_infos["raceDate"].dt.strftime("%Y-%m-%d") + " " + dog_infos["raceTime"].astype(str),
    errors="coerce",
)
dog_infos["raceClass"] = dog_infos["raceClass"].astype("string").str.strip().str.upper()
dog_infos["resultPosition"] = pd.to_numeric(dog_infos["resultPosition"], errors="coerce")
dog_infos["resultRunTime"] = pd.to_numeric(dog_infos["resultRunTime"], errors="coerce")

dog_infos = dog_infos.loc[
    dog_infos["dogId"].notna()
    & dog_infos["raceId"].notna()
    & dog_infos["raceDateTime"].notna()
    & dog_infos["resultPosition"].notna()
].copy()

dog_infos["dogId"] = dog_infos["dogId"].astype(int)
dog_infos["raceId"] = dog_infos["raceId"].astype(int)
dog_infos["trapNumber"] = pd.to_numeric(dog_infos["trapNumber"], errors="coerce")

dog_infos.shape

(3383103, 13)

## 6.2 Helper functions

The implementation below includes:

- distance bands
- starting-price implied probability seeding
- time-based rating decay
- pairwise expected score
- margin-aware actual score
- race-level sequential updates

In [3]:
def distance_band(distance):
    if pd.isna(distance):
        return "unknown"
    if distance <= 400:
        return "sprint"
    if distance <= 550:
        return "middle"
    return "staying"


def implied_probability(numerator, denominator):
    if pd.isna(numerator) or pd.isna(denominator) or denominator == 0:
        return np.nan
    decimal_price = 1 + (numerator / denominator)
    if decimal_price <= 1:
        return np.nan
    return 1 / decimal_price


def seed_rating_from_market(probability, base_rating=1500.0):
    if pd.isna(probability):
        return base_rating
    probability = float(np.clip(probability, 0.05, 0.95))
    return base_rating + 400.0 * math.log10(probability / (1.0 - probability))


def decayed_rating(current_rating, last_race_dt, current_race_dt, base_rating=1500.0, decay_lambda=0.003):
    if last_race_dt is None or pd.isna(last_race_dt) or pd.isna(current_race_dt):
        return current_rating
    days_since_last_race = max((current_race_dt - last_race_dt).days, 0)
    decay_weight = math.exp(-decay_lambda * days_since_last_race)
    return base_rating + (current_rating - base_rating) * decay_weight


def expected_score(rating_a, rating_b):
    return 1.0 / (1.0 + 10.0 ** ((rating_b - rating_a) / 400.0))


def actual_pairwise_score(row_a, row_b, margin_sigma=0.20):
    pos_a = row_a.resultPosition
    pos_b = row_b.resultPosition

    if pos_a < pos_b:
        binary_score = 1.0
    elif pos_a > pos_b:
        binary_score = 0.0
    else:
        binary_score = 0.5

    time_a = row_a.resultRunTime
    time_b = row_b.resultRunTime
    if pd.isna(time_a) or pd.isna(time_b):
        return binary_score

    time_gap = time_b - time_a
    margin_score = 0.5 + 0.5 * np.tanh(time_gap / margin_sigma)

    if binary_score == 1.0:
        return max(0.5, margin_score)
    if binary_score == 0.0:
        return min(0.5, margin_score)
    return 0.5


def field_win_probabilities(effective_ratings):
    strengths = np.power(10.0, np.asarray(effective_ratings) / 400.0)
    return strengths / strengths.sum()


## 6.3 Sequential greyhound ELO engine

The race update is done in chronological order. For each race, we:

1. compute pre-race decayed ratings
2. combine overall and distance-band ratings into an effective pre-race rating
3. compute all pairwise expected and actual scores
4. average the pairwise updates back into a single per-dog race update
5. store pre-race and post-race ratings on each race row

In [4]:
def compute_greyhound_elo(
    dog_infos,
    base_rating=1500.0,
    k_factor=20.0,
    decay_lambda=0.003,
    margin_sigma=0.20,
    overall_weight=0.60,
    distance_weight=0.40,
    seed_from_sp=True,
):
    df = dog_infos.copy()
    df["distanceBand"] = df["raceDistance"].apply(distance_band)
    df["spImpliedProb"] = df.apply(
        lambda row: implied_probability(row["numeratorSP"], row["denominatorSP"]),
        axis=1,
    )

    df = df.sort_values(["raceDateTime", "raceId", "resultPosition", "dogId"]).copy()

    overall_ratings = {}
    distance_ratings = {}
    last_overall_race_dt = {}
    last_distance_race_dt = {}

    enriched_races = []

    for race_id, race in df.groupby("raceId", sort=False):
        race = race.sort_values(["resultPosition", "resultRunTime", "dogId"]).copy()
        race_dt = race["raceDateTime"].iloc[0]
        field_size = len(race)

        overall_pre = []
        distance_pre = []
        effective_pre = []

        for row in race.itertuples():
            dog_id = row.dogId
            band = row.distanceBand

            if dog_id not in overall_ratings:
                overall_ratings[dog_id] = (
                    seed_rating_from_market(row.spImpliedProb, base_rating)
                    if seed_from_sp else base_rating
                )

            if (dog_id, band) not in distance_ratings:
                distance_ratings[(dog_id, band)] = overall_ratings[dog_id]

            current_overall = decayed_rating(
                current_rating=overall_ratings[dog_id],
                last_race_dt=last_overall_race_dt.get(dog_id),
                current_race_dt=race_dt,
                base_rating=base_rating,
                decay_lambda=decay_lambda,
            )

            current_distance = decayed_rating(
                current_rating=distance_ratings[(dog_id, band)],
                last_race_dt=last_distance_race_dt.get((dog_id, band)),
                current_race_dt=race_dt,
                base_rating=base_rating,
                decay_lambda=decay_lambda,
            )

            current_effective = overall_weight * current_overall + distance_weight * current_distance

            overall_pre.append(current_overall)
            distance_pre.append(current_distance)
            effective_pre.append(current_effective)

        expected_totals = np.zeros(field_size)
        actual_totals = np.zeros(field_size)

        race_rows = list(race.itertuples())
        for i in range(field_size):
            for j in range(i + 1, field_size):
                exp_i = expected_score(effective_pre[i], effective_pre[j])
                act_i = actual_pairwise_score(race_rows[i], race_rows[j], margin_sigma=margin_sigma)

                expected_totals[i] += exp_i
                expected_totals[j] += (1.0 - exp_i)
                actual_totals[i] += act_i
                actual_totals[j] += (1.0 - act_i)

        race_update = k_factor * (actual_totals - expected_totals) / max(field_size - 1, 1)
        overall_post = np.asarray(overall_pre) + race_update
        distance_post = np.asarray(distance_pre) + race_update
        effective_post = overall_weight * overall_post + distance_weight * distance_post
        win_probs = field_win_probabilities(effective_pre)

        race["eloOverallPre"] = overall_pre
        race["eloDistancePre"] = distance_pre
        race["eloEffectivePre"] = effective_pre
        race["eloExpectedPairwise"] = expected_totals / max(field_size - 1, 1)
        race["eloActualPairwise"] = actual_totals / max(field_size - 1, 1)
        race["eloRaceDelta"] = race_update
        race["eloOverallPost"] = overall_post
        race["eloDistancePost"] = distance_post
        race["eloEffectivePost"] = effective_post
        race["eloWinProbPre"] = win_probs
        race["eloFieldSize"] = field_size
        race["eloFieldMeanPre"] = np.mean(effective_pre)
        race["eloVsFieldMeanPre"] = race["eloEffectivePre"] - race["eloFieldMeanPre"]

        for row, overall_new, distance_new in zip(race.itertuples(), overall_post, distance_post):
            dog_id = row.dogId
            band = row.distanceBand
            overall_ratings[dog_id] = overall_new
            distance_ratings[(dog_id, band)] = distance_new
            last_overall_race_dt[dog_id] = race_dt
            last_distance_race_dt[(dog_id, band)] = race_dt

        enriched_races.append(race)

    elo_df = pd.concat(enriched_races, axis=0).sort_index()
    return elo_df

## 6.4 Run the ELO pipeline

These defaults are a good first baseline:

- `base_rating=1500`
- `k_factor=20`
- `decay_lambda=0.003`
- `margin_sigma=0.20`
- `overall_weight=0.60`
- `distance_weight=0.40`

You can tune these later against a chronological validation split.

In [5]:
dog_infos_elo = compute_greyhound_elo(
    dog_infos,
    base_rating=1500.0,
    k_factor=20.0,
    decay_lambda=0.003,
    margin_sigma=0.20,
    overall_weight=0.60,
    distance_weight=0.40,
    seed_from_sp=True,
)

dog_infos_elo.shape

KeyboardInterrupt: 

## 6.5 Inspect the new ELO features

In [ ]:
elo_feature_cols = [
    "dogId",
    "raceId",
    "raceDateTime",
    "trackName",
    "raceDistance",
    "distanceBand",
    "resultPosition",
    "eloOverallPre",
    "eloDistancePre",
    "eloEffectivePre",
    "eloWinProbPre",
    "eloRaceDelta",
    "eloOverallPost",
]

dog_infos_elo[elo_feature_cols].head(20)

In [ ]:
dog_infos_elo[[
    "eloOverallPre",
    "eloDistancePre",
    "eloEffectivePre",
    "eloWinProbPre",
    "eloRaceDelta",
]].describe()

## 6.6 Save the enriched dataset

In [ ]:
dog_infos_elo.to_csv("../data/intermediate/06_dog_infos_with_elo.csv", index=False)

## 6.7 Recommended next improvements

As a greyhound racing modeler, I would validate and extend this ELO in the following order:

1. Tune `k_factor`, `decay_lambda`, and `margin_sigma` on a time-based validation split.
2. Compare plain overall ELO vs overall-plus-distance ELO.
3. Add expanding-history track/trap bias corrections, avoiding future leakage.
4. Compare ELO implied win probabilities to market implied probabilities.
5. Feed pre-race ELO features into the sectional-time or finishing-position model.